In [2]:
import pandas as pd

In [3]:
listings = pd.read_csv("../data/raw/listings2.csv")

In [ ]:
import pandas as pd

listings = pd.read_csv("../data/raw/listings2.csv")
calendar = pd.read_csv("../data/raw/calendar.csv", parse_dates=["date"])
reviews = pd.read_csv("../data/raw/reviews2.csv", parse_dates=["date"])
neighbourhoods = pd.read_csv("../data/raw/neighbourhoods.csv")

##3.1 Data Inges on & Profiling 

#Turn loading code into a repeatable ingestion function

In [8]:
import pandas as pd
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger("ingestion")

def load_csv(filename, parse_dates=None, data_dir="../data/raw"):
    path = Path(data_dir) / filename
    try:
        df = pd.read_csv(path, parse_dates=parse_dates)
        logger.info(f"Loaded {filename}: {df.shape[0]:,} rows x {df.shape[1]} cols")
        return df
    except FileNotFoundError:
        logger.error(f"File not found: {path}")
        raise
    except Exception as e:
        logger.error(f"Failed to load {filename}: {e}")
        raise

listings = load_csv("listings2.csv")
calendar = load_csv("calendar.csv", parse_dates=["date"])
reviews = load_csv("reviews2.csv", parse_dates=["date"])
neighbourhoods = load_csv("neighbourhoods.csv")

2026-07-11 06:27:49,320 | INFO | Loaded listings2.csv: 30,259 rows x 90 cols
2026-07-11 06:28:03,952 | INFO | Loaded calendar.csv: 11,152,576 rows x 5 cols
2026-07-11 06:28:12,921 | INFO | Loaded reviews2.csv: 990,170 rows x 6 cols
2026-07-11 06:28:12,930 | INFO | Loaded neighbourhoods.csv: 230 rows x 2 cols


#Full profiling report (row counts, cardinality, null rates, dtype distributions)

In [9]:
def profile_dataframe(df, name):
    profile = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_unique': df.nunique(),
        'n_missing': df.isnull().sum(),
        'pct_missing': (df.isnull().mean() * 100).round(2),
    })
    profile.insert(0, 'dataset', name)
    return profile.reset_index().rename(columns={'index': 'column'})

datasets = {'listings': listings, 'calendar': calendar, 'reviews': reviews, 'neighbourhoods': neighbourhoods}
profiles = pd.concat([profile_dataframe(df, name) for name, df in datasets.items()], ignore_index=True)

profiles.to_csv("../docs/data_quality_report.csv", index=False)
profiles[profiles['pct_missing'] > 0].sort_values('pct_missing', ascending=False)

,column,dataset,dtype,n_unique,n_missing,pct_missing
7,neighborhood_overview,listings,float64,0,30259,100.00
14,host_since,listings,float64,0,30259,100.00
29,host_total_listings_count,listings,float64,0,30259,100.00
33,neighbourhood,listings,float64,0,30259,100.00
30,host_verifications,listings,float64,0,30259,100.00
22,host_response_rate,listings,float64,0,30259,100.00
23,host_acceptance_rate,listings,float64,0,30259,100.00
25,host_thumbnail_url,listings,float64,0,30259,100.00
21,host_response_time,listings,float64,0,30259,100.00
27,host_neighbourhood,listings,float64,0,30259,100.00


#Duplicate detection: deterministic (done) + fuzzy (new)

In [10]:
from rapidfuzz import fuzz

In [11]:
# Compare listing names within the same host — near-duplicate check
def find_fuzzy_duplicates(df, host_col='host_id', text_col='name', threshold=90):
    suspects = []
    for host_id, group in df.groupby(host_col):
        if len(group) < 2:
            continue
        names = group[[text_col, 'id']].dropna()
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                score = fuzz.ratio(names.iloc[i][text_col], names.iloc[j][text_col])
                if score >= threshold:
                    suspects.append((names.iloc[i]['id'], names.iloc[j]['id'], score))
    return pd.DataFrame(suspects, columns=['id_1', 'id_2', 'similarity'])

fuzzy_dupes = find_fuzzy_duplicates(listings)
print(f"Potential near-duplicate listings: {len(fuzzy_dupes)}")

Potential near-duplicate listings: 10656


#Completeness assessment

In [12]:
key_completeness = profiles[profiles['dataset'] == 'listings'].nlargest(10, 'pct_missing')
print(key_completeness[['column', 'pct_missing']])

                       column  pct_missing
7       neighborhood_overview        100.0
14                 host_since        100.0
21         host_response_time        100.0
22         host_response_rate        100.0
23       host_acceptance_rate        100.0
25         host_thumbnail_url        100.0
27         host_neighbourhood        100.0
29  host_total_listings_count        100.0
30         host_verifications        100.0
33              neighbourhood        100.0


#Outlier detection on key numerical fields

In [13]:
from scipy import stats

In [14]:
def outlier_summary(series, name):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = series[(series < lower) | (series > upper)]
    print(f"{name}: {len(outliers)} outliers ({len(outliers)/len(series)*100:.1f}%), "
          f"range beyond [{lower:.1f}, {upper:.1f}]")

# price is likely still a string with $ — check first
print(listings['price'].dtype, listings['price'].head())

str 0    $113.97
1    $117.27
2     $80.06
3     $77.17
4    $202.47
Name: price, dtype: str


#Domain validation rules

In [15]:
if pd.api.types.is_numeric_dtype(listings['price']):
    negative_price = (listings['price'] < 0).sum()
else:
    negative_price = (
        listings['price'].astype(str)
        .str.replace(r'[$,]', '', regex=True)
        .astype(float) < 0
    ).sum()

validations = {
    'negative_price': negative_price,
    'invalid_latitude': (~listings['latitude'].between(-90, 90)).sum(),
    'invalid_longitude': (~listings['longitude'].between(-180, 180)).sum(),
    'min_nights_gt_max': (listings['minimum_nights'] > listings['maximum_nights']).sum(),
    'availability_out_of_range': (~listings['availability_365'].between(0, 365)).sum(),
}
pd.Series(validations)

negative_price               0
invalid_latitude             0
invalid_longitude            0
min_nights_gt_max            0
availability_out_of_range    0
dtype: int64

In [16]:
print(fuzzy_dupes.head(10))
sample_ids = fuzzy_dupes.iloc[0][['id_1','id_2']]
listings[listings['id'].isin(sample_ids)][['id','host_id','name']]

       id_1                 id_2  similarity
0  47803519             51455566   96.000000
1  47803519             51455941   96.000000
2  47803519             51455950   96.000000
3  51455566             51455941   96.000000
4  51455566             51455950   96.000000
5  51455941             51455950   96.000000
6     31130             20942873   95.000000
7   5237336  1591622583865268747   90.322581
8    294227               294242  100.000000
9    294227               294259  100.000000


,id,host_id,name
11481,47803519,36641,Historic brownstone - Rm1
12516,51455566,36641,Historic brownstone - Rm2


In [17]:
orphans_cal = set(calendar['listing_id']) - set(listings['id'])
orphans_rev = set(reviews['listing_id']) - set(listings['id'])
print(f"Calendar orphans: {len(orphans_cal)}")
print(f"Review orphans: {len(orphans_rev)}")

Calendar orphans: 296
Review orphans: 239


In [18]:
# Calendar checks
print(calendar['available'].unique())
print(f"Calendar date range: {calendar['date'].min()} to {calendar['date'].max()}")

def outlier_summary(series, name):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = series[(series < lower) | (series > upper)]
    print(f"{name}: {len(outliers):,} outliers ({len(outliers)/len(series)*100:.2f}%), "
          f"IQR bounds [{lower:.1f}, {upper:.1f}], actual range [{series.min()}, {series.max()}]")

outlier_summary(calendar['minimum_nights'], 'calendar_minimum_nights')
outlier_summary(calendar['maximum_nights'], 'calendar_maximum_nights')

# Reviews checks
review_counts = reviews.groupby('listing_id').size()
outlier_summary(review_counts, 'reviews_per_listing')
print(f"Max reviews for a single listing: {review_counts.max()}")
print(f"Listings with 1 review: {(review_counts == 1).sum()}")
print(f"Listings with 100+ reviews: {(review_counts >= 100).sum()}")

<ArrowStringArray>
['f', 't']
Length: 2, dtype: str
Calendar date range: 2026-06-14 00:00:00 to 2027-06-22 00:00:00
calendar_minimum_nights: 2,503,219 outliers (22.45%), IQR bounds [30.0, 30.0], actual range [1, 1124]
calendar_maximum_nights: 7,035 outliers (0.06%), IQR bounds [-1237.5, 2542.5], actual range [1, 2147483647]
reviews_per_listing: 2,320 outliers (10.57%), IQR bounds [-67.5, 120.5], actual range [1, 4502]
Max reviews for a single listing: 4502
Listings with 1 review: 2771
Listings with 100+ reviews: 2967


In [19]:
print(calendar['minimum_nights'].value_counts().head())

minimum_nights
30    8649357
1     1201232
2      381370
3      290154
31     124009
Name: count, dtype: int64


In [20]:
print((calendar['maximum_nights'] == 2147483647).sum())
print(calendar['maximum_nights'].value_counts().head(10))

5210
maximum_nights
1125    4288515
365     3333138
30       460593
90       387895
28       360448
60       308504
180      191335
730      187129
120      139225
31       103868
Name: count, dtype: int64


In [21]:
top_listing = review_counts.idxmax()
print(listings[listings['id'] == top_listing][['id', 'name', 'first_review', 'last_review', 'number_of_reviews']])

                       id                                            name  \
18410  858697692672545141  Safe and Cozy Hostel Room, 1 person, Manhattan   

      first_review last_review  number_of_reviews  
18410   2023-04-04  2026-06-12               4502  


##3.2  Data Cleaning & Standardiza on 

In [22]:
#Drop the 12 dead columns

In [23]:
dead_columns = [
    'neighborhood_overview', 'host_since', 'host_response_time',
    'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url',
    'host_neighbourhood', 'host_total_listings_count', 'host_verifications',
    'neighbourhood', 'calendar_updated', 'instant_bookable'
]

listings_clean = listings.drop(columns=dead_columns)
print(f"Dropped {len(dead_columns)} columns. Shape: {listings.shape} -> {listings_clean.shape}")

Dropped 12 columns. Shape: (30259, 90) -> (30259, 78)


#Standardize price columns

In [24]:
def clean_price(series):
    return (
        series.astype(str)
        .str.replace(r'[$,]', '', regex=True)
        .replace('nan', pd.NA)
        .astype(float)
    )

price_cols = ['price', 'price_quote_total_price', 'price_quote_price_per_night']
for col in price_cols:
    listings_clean[col] = clean_price(listings_clean[col])

print(listings_clean[price_cols].describe())

              price  price_quote_total_price  price_quote_price_per_night
count  21515.000000             21514.000000                 21514.000000
mean     278.229518              5440.077490                   278.232165
std      531.730477             12598.411711                   531.742693
min        4.580000                69.000000                     4.580000
25%       96.875000              1546.520000                    96.872500
50%      174.690000              3274.280000                   174.680000
75%      302.720000              6261.500000                   302.745000
max    30972.960000            929188.680000                 30972.960000


#Fix the maximum_nights sentinel overflow in calendar

In [25]:
calendar_clean = calendar.copy()
SENTINEL = 2147483647
calendar_clean.loc[calendar_clean['maximum_nights'] == SENTINEL, 'maximum_nights'] = pd.NA
print(f"Nulled {(calendar['maximum_nights'] == SENTINEL).sum()} sentinel rows in calendar")

Nulled 5210 sentinel rows in calendar


In [26]:
for col in ['maximum_nights', 'maximum_maximum_nights', 'maximum_nights_avg_ntm']:
    n_sentinel = (listings_clean[col] >= SENTINEL).sum()
    print(f"{col}: {n_sentinel} sentinel/overflow values")

maximum_nights: 26 sentinel/overflow values
maximum_maximum_nights: 26 sentinel/overflow values
maximum_nights_avg_ntm: 13 sentinel/overflow values


#Handle the "no reviews yet" cluster 

In [27]:
listings_clean['has_reviews'] = listings_clean['number_of_reviews'] > 0
print(listings_clean['has_reviews'].value_counts())

# Sanity check: does has_reviews line up with the missing review_scores_rating?
print(pd.crosstab(listings_clean['has_reviews'], listings_clean['review_scores_rating'].isna()))

has_reviews
True     21700
False     8559
Name: count, dtype: int64
review_scores_rating  False  True 
has_reviews                       
False                     0   8559
True                  21700      0


#Parse date fields

In [28]:
date_cols = ['first_review', 'last_review']
for col in date_cols:
    listings_clean[col] = pd.to_datetime(listings_clean[col], errors='coerce')

print(listings_clean[date_cols].dtypes)
print(listings_clean[date_cols].describe())

first_review    datetime64[us]
last_review     datetime64[us]
dtype: object
                     first_review                 last_review
count                       21700                       21700
mean   2021-01-14 01:06:57.400921  2023-12-11 12:51:25.714285
min           2009-05-25 00:00:00         2011-05-12 00:00:00
25%           2018-03-29 00:00:00         2023-01-01 00:00:00
50%           2021-12-19 00:00:00         2025-04-21 12:00:00
75%           2023-11-03 06:00:00         2026-04-18 00:00:00
max           2026-06-14 00:00:00         2026-06-22 00:00:00


#Null the small sentinel counts in listings_clean

In [29]:
listings_clean.loc[listings_clean['maximum_nights'] >= SENTINEL, 'maximum_nights'] = pd.NA
listings_clean.loc[listings_clean['maximum_maximum_nights'] >= SENTINEL, 'maximum_maximum_nights'] = pd.NA
listings_clean.loc[listings_clean['maximum_nights_avg_ntm'] >= SENTINEL, 'maximum_nights_avg_ntm'] = pd.NA

print("Sentinel values nulled in listings_clean")

Sentinel values nulled in listings_clean


#Standardize categorical fields

In [30]:
print(listings_clean['room_type'].value_counts())
print()
print(listings_clean['property_type'].value_counts())

room_type
Entire home/apt    16808
Private room       12713
Hotel room           520
Shared room          218
Name: count, dtype: int64

property_type
Entire rental unit                  13107
Private room in rental unit          6858
Private room in home                 2482
Room in hotel                        1681
Entire home                          1222
                                    ...  
Private room in tower                   1
Room in bed and breakfast               1
Shared room in bed and breakfast        1
Private room in cave                    1
Castle                                  1
Name: count, Length: 69, dtype: int64


#Standardize geographic fields

In [31]:
mismatch = set(listings_clean['neighbourhood_cleansed']) - set(neighbourhoods['neighbourhood'])
print(f"Neighbourhoods in listings but not in neighbourhoods.csv: {len(mismatch)}")
print(mismatch)

Neighbourhoods in listings but not in neighbourhoods.csv: 0
set()


In [32]:
print(listings_clean['neighbourhood_group_cleansed'].value_counts())

neighbourhood_group_cleansed
Manhattan        13709
Brooklyn         10562
Queens            4680
Bronx              976
Staten Island      332
Name: count, dtype: int64


In [33]:
#build a simplified property_type_group

In [34]:
def simplify_property_type(pt):
    pt_lower = str(pt).lower()
    if 'entire' in pt_lower:
        return 'Entire place'
    elif 'private room' in pt_lower:
        return 'Private room'
    elif 'shared room' in pt_lower:
        return 'Shared room'
    elif 'hotel' in pt_lower or 'room in' in pt_lower:
        return 'Hotel/B&B'
    else:
        return 'Other'

listings_clean['property_type_group'] = listings_clean['property_type'].apply(simplify_property_type)
print(listings_clean['property_type_group'].value_counts())
print()
# Sanity check: does this roughly track room_type?
print(pd.crosstab(listings_clean['property_type_group'], listings_clean['room_type']))

property_type_group
Entire place    16703
Private room    11295
Hotel/B&B        1995
Shared room       218
Other              48
Name: count, dtype: int64

room_type            Entire home/apt  Hotel room  Private room  Shared room
property_type_group                                                        
Entire place                   16703           0             0            0
Hotel/B&B                         57         520          1418            0
Other                             48           0             0            0
Private room                       0           0         11295            0
Shared room                        0           0             0          218


#Save cleaned data to Parquet

In [35]:
!pip install pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [36]:
!pip install --upgrade pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [37]:
from pathlib import Path

interim_dir = Path("../data/interim")
interim_dir.mkdir(parents=True, exist_ok=True)

listings_clean.to_parquet(interim_dir / "listings_silver.parquet", index=False)
calendar_clean.to_parquet(interim_dir / "calendar_silver.parquet", index=False)
reviews.to_parquet(interim_dir / "reviews_silver.parquet", index=False)
neighbourhoods.to_parquet(interim_dir / "neighbourhoods_silver.parquet", index=False)

print("Saved silver-layer Parquet files:")
for f in interim_dir.glob("*_silver.parquet"):
    print(f" - {f.name}: {f.stat().st_size / 1024**2:.1f} MB")

Saved silver-layer Parquet files:
 - calendar_silver.parquet: 4.8 MB
 - listings_silver.parquet: 18.5 MB
 - neighbourhoods_silver.parquet: 0.0 MB
 - reviews_silver.parquet: 160.9 MB


In [38]:
!pip install duckdb

Defaulting to user installation because normal site-packages is not writeable


In [39]:
import duckdb

con = duckdb.connect(database=':memory:')

con.register('listings', listings_clean)
con.register('calendar', calendar_clean)
con.register('reviews', reviews)
con.register('neighbourhoods', neighbourhoods)

con.execute("SELECT COUNT(*) FROM calendar").fetchdf()

,count_star()
0,11152576


#Join listings with review summaries

In [40]:
review_summary = con.execute("""
    SELECT
        listing_id,
        COUNT(*) AS review_count_actual,
        MIN(date) AS first_review_date_actual,
        MAX(date) AS last_review_date_actual
    FROM reviews
    GROUP BY listing_id
""").fetchdf()

print(review_summary.shape)
review_summary.head()

(21939, 4)


,listing_id,review_count_actual,first_review_date_actual,last_review_date_actual
0,6848,198,2009-05-25,2026-04-23
1,263005,129,2012-01-03,2023-08-30
2,6872,2,2022-06-05,2025-10-07
3,6990,251,2009-10-28,2026-01-11
4,118061,100,2011-10-24,2026-05-30


#join it into the listings master table

In [41]:
listings_enriched = con.execute("""
    SELECT
        l.*,
        r.review_count_actual,
        r.first_review_date_actual,
        r.last_review_date_actual
    FROM listings l
    LEFT JOIN review_summary r ON l.id = r.listing_id
""").fetchdf()

print(listings_enriched.shape)

(30259, 83)


#cross-validates self-reported vs. actual review counts

In [42]:
mismatch = listings_enriched[
    listings_enriched['number_of_reviews'] != listings_enriched['review_count_actual'].fillna(0)
]
print(f"Listings where reported count != actual count: {len(mismatch)}")

Listings where reported count != actual count: 0


#Calendar occupancy & revenue estimation

In [43]:
occupancy_revenue = con.execute("""
    SELECT
        c.listing_id,
        COUNT(*) AS total_days_tracked,
        SUM(CASE WHEN c.available = 'f' THEN 1 ELSE 0 END) AS booked_days,
        ROUND(
            SUM(CASE WHEN c.available = 'f' THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 4
        ) AS occupancy_rate
    FROM calendar c
    GROUP BY c.listing_id
""").fetchdf()

print(occupancy_revenue.shape)
occupancy_revenue.head()

(30555, 4)


,listing_id,total_days_tracked,booked_days,occupancy_rate
0,852118,365,185.0,0.5068
1,649972,365,365.0,1.0000
2,856918,365,8.0,0.0219
3,867020,365,18.0,0.0493
4,657198,365,365.0,1.0000


#bring in price to estimate revenue, joining against your enriched listings

In [44]:
con.register('occupancy_revenue', occupancy_revenue)
con.register('listings_enriched', listings_enriched)

listings_with_revenue = con.execute("""
    SELECT
        le.*,
        o.total_days_tracked,
        o.booked_days,
        o.occupancy_rate,
        ROUND(o.booked_days * le.price, 2) AS estimated_revenue
    FROM listings_enriched le
    LEFT JOIN occupancy_revenue o ON le.id = o.listing_id
""").fetchdf()

print(listings_with_revenue.shape)
listings_with_revenue[['id', 'price', 'occupancy_rate', 'estimated_revenue']].head()

(30259, 87)


,id,price,occupancy_rate,estimated_revenue
0,2539,113.97,0.0712,2963.22
1,6848,117.27,0.4767,20404.98
2,6872,80.06,0.2164,6324.74
3,6990,77.17,0.4027,11343.99
4,7097,202.47,0.7096,52439.73


In [ ]:
#Neighbourhood-level aggregates

In [45]:
con.register('listings_with_revenue', listings_with_revenue)

In [46]:
neighbourhood_stats = con.execute("""
    SELECT
        neighbourhood_cleansed,
        neighbourhood_group_cleansed,
        COUNT(*) AS listing_count,
        MEDIAN(price) AS median_price,
        ROUND(AVG(review_scores_rating), 3) AS avg_rating,
        ROUND(AVG(occupancy_rate), 4) AS avg_occupancy_rate
    FROM listings_with_revenue
    GROUP BY neighbourhood_cleansed, neighbourhood_group_cleansed
    ORDER BY listing_count DESC
""").fetchdf()

print(neighbourhood_stats.shape)
neighbourhood_stats.head(10)

(223, 6)


,neighbourhood_cleansed,neighbourhood_group_cleansed,listing_count,median_price,avg_rating,avg_occupancy_rate
0,Bedford-Stuyvesant,Brooklyn,2089,133.775,4.740,0.5099
1,Midtown,Manhattan,1784,378.720,4.623,0.4272
2,Williamsburg,Brooklyn,1516,187.610,4.813,0.6809
3,Harlem,Manhattan,1417,117.840,4.741,0.4823
4,Hell's Kitchen,Manhattan,1365,234.170,4.674,0.4279
5,Upper West Side,Manhattan,1194,206.500,4.722,0.4757
6,Upper East Side,Manhattan,1187,229.790,4.673,0.3941
7,Bushwick,Brooklyn,1016,112.740,4.758,0.5572
8,Crown Heights,Brooklyn,876,148.080,4.775,0.5189
9,Chelsea,Manhattan,733,238.820,4.695,0.4855


In [47]:
con.register('neighbourhood_stats', neighbourhood_stats)

listings_final = con.execute("""
    SELECT
        lwr.*,
        ns.listing_count AS neighbourhood_listing_count,
        ns.median_price AS neighbourhood_median_price,
        ns.avg_rating AS neighbourhood_avg_rating,
        ns.avg_occupancy_rate AS neighbourhood_avg_occupancy
    FROM listings_with_revenue lwr
    LEFT JOIN neighbourhood_stats ns
        ON lwr.neighbourhood_cleansed = ns.neighbourhood_cleansed
        AND lwr.neighbourhood_group_cleansed = ns.neighbourhood_group_cleansed
""").fetchdf()

print(listings_final.shape)

(30259, 91)


#Derived calculated fields

In [48]:
import datetime

In [49]:
# Host tenure in years — using host_id first-seen date isn't directly available,
# but we can approximate host tenure from their earliest listing's first review,
# or more simply from host_listings_count context. Since host_since was 100% missing
# (dropped in §3.2), we'll derive tenure from the earliest first_review per host instead.

host_tenure = con.execute("""
    SELECT
        host_id,
        MIN(first_review) AS host_earliest_review
    FROM listings_final
    WHERE first_review IS NOT NULL
    GROUP BY host_id
""").fetchdf()

con.register('host_tenure', host_tenure)

listings_final2 = con.execute("""
    SELECT
        lf.*,
        ht.host_earliest_review,
        DATE_DIFF('year', ht.host_earliest_review, CURRENT_DATE) AS host_tenure_years_est,
        ROUND(lf.review_count_actual * 1.0 /
            NULLIF(DATE_DIFF('month', lf.first_review_date_actual, lf.last_review_date_actual), 0), 3
        ) AS review_frequency_per_month,
        ROUND(lf.price / NULLIF(lf.bedrooms, 0), 2) AS price_per_bedroom
    FROM listings_final lf
    LEFT JOIN host_tenure ht ON lf.host_id = ht.host_id
""").fetchdf()

print(listings_final2.shape)
listings_final2[['host_id', 'host_tenure_years_est', 'review_frequency_per_month', 'price_per_bedroom']].describe()

(30259, 95)


,host_id,host_tenure_years_est,review_frequency_per_month,price_per_bedroom
count,3.025900e+04,26020.0,18655.000000,13966.000000
mean,2.672815e+15,6.109454,1.514960,229.217174
std,6.705743e+16,3.58321,2.229819,367.666427
min,1.678000e+03,0.0,0.014000,2.400000
25%,2.069159e+07,3.0,0.333000,90.872500
50%,1.074344e+08,6.0,0.875000,152.625000
75%,3.948700e+08,9.0,2.000000,265.500000
max,1.704453e+18,17.0,118.474000,10986.000000


In [50]:
print(listings_final2['host_id'].dtype)
suspicious = listings_final2[listings_final2['host_id'] > 1e12]
print(suspicious[['id', 'host_id', 'name']] if 'name' in suspicious.columns else suspicious[['id','host_id']])

int64
                        id              host_id  \
28838  1671232927924501778  1671232802933013831   
28859  1679757208951712556  1679756815269193200   
28863  1700185373628753962  1700185117887352466   
29679  1668518798332888795  1668507700349210879   
29682  1668843503888628364  1668843439940951391   
29684  1663370093718129946  1663370027188720324   
29723  1666751797770083633  1666751745677666551   
29739  1670651555776768637  1670651389176408819   
29759  1672874147215482492  1672873886589183474   
29763  1673484968022802209  1673484849367509139   
29768  1673569350477013917  1672891640562272393   
29772  1675017801355776854  1675017754018998796   
29773  1675046024335034716  1675017754018998796   
29797  1677549264647718691  1677548934238826298   
29799  1677883930809390661  1677883864710641096   
29812  1688086444022224211  1688083961939378163   
29833  1678566825758572131  1678566152189001338   
29836  1682053381122292502  1682052323187232784   
29838  168851105888618484

In [51]:
print(listings_final2.nlargest(5, 'price_per_bedroom')[['id', 'price', 'bedrooms', 'price_per_bedroom']])

                        id    price  bedrooms  price_per_bedroom
28583  1614018340350324380  10986.0       1.0            10986.0
28584  1614018632211285219  10986.0       1.0            10986.0
28585  1614018927799603136  10986.0       1.0            10986.0
28586  1614020137045877415  10986.0       1.0            10986.0
9225              39553889  10000.0       1.0            10000.0


#3.4 Data Modeling 

In [ ]:
#Switch to a persistent DuckDB file

In [52]:
con.close()  # close the in-memory connection

con = duckdb.connect(database='../data/airbnb_warehouse.duckdb')

# Re-register your final enriched table and the neighbourhood stats
con.register('listings_final2', listings_final2)
con.register('neighbourhood_stats', neighbourhood_stats)

#Create the dimension tables

In [53]:
con.execute("""
    CREATE OR REPLACE TABLE dim_listing AS
    SELECT DISTINCT
        id AS listing_id,
        property_type,
        property_type_group,
        room_type,
        accommodates,
        bedrooms,
        bathrooms,
        neighbourhood_cleansed,
        neighbourhood_group_cleansed,
        latitude,
        longitude,
        first_review_date_actual AS first_review,
        last_review_date_actual AS last_review
    FROM listings_final2
""")

con.execute("""
    CREATE OR REPLACE TABLE dim_host AS
    SELECT DISTINCT
        host_id,
        host_is_superhost,
        host_listings_count,
        host_tenure_years_est,
        calculated_host_listings_count
    FROM listings_final2
""")

con.execute("""
    CREATE OR REPLACE TABLE dim_neighbourhood AS
    SELECT
        neighbourhood_cleansed,
        neighbourhood_group_cleansed,
        listing_count AS neighbourhood_listing_count,
        median_price AS neighbourhood_median_price,
        avg_rating AS neighbourhood_avg_rating,
        avg_occupancy_rate AS neighbourhood_avg_occupancy
    FROM neighbourhood_stats
""")

print(con.execute("SELECT COUNT(*) FROM dim_listing").fetchdf())
print(con.execute("SELECT COUNT(*) FROM dim_host").fetchdf())
print(con.execute("SELECT COUNT(*) FROM dim_neighbourhood").fetchdf())

   count_star()
0         30259
   count_star()
0         16474
   count_star()
0           223


#Create the fact table

In [54]:
con.execute("""
    CREATE OR REPLACE TABLE fact_listing_performance AS
    SELECT
        id AS listing_id,
        host_id,
        neighbourhood_cleansed,
        neighbourhood_group_cleansed,
        price,
        bedrooms,
        price_per_bedroom,
        occupancy_rate,
        booked_days,
        total_days_tracked,
        estimated_revenue,
        review_count_actual,
        review_frequency_per_month,
        review_scores_rating,
        has_reviews
    FROM listings_final2
""")

print(con.execute("SELECT COUNT(*) FROM fact_listing_performance").fetchdf())

   count_star()
0         30259


#Verify the model with a real analytical query

In [55]:
result = con.execute("""
    SELECT
        dn.neighbourhood_group_cleansed AS borough,
        dl.room_type,
        COUNT(*) AS num_listings,
        ROUND(AVG(f.price), 2) AS avg_price,
        ROUND(AVG(f.occupancy_rate), 3) AS avg_occupancy,
        ROUND(AVG(f.estimated_revenue), 2) AS avg_est_revenue
    FROM fact_listing_performance f
    JOIN dim_listing dl ON f.listing_id = dl.listing_id
    JOIN dim_neighbourhood dn ON f.neighbourhood_cleansed = dn.neighbourhood_cleansed
    GROUP BY dn.neighbourhood_group_cleansed, dl.room_type
    ORDER BY borough, avg_est_revenue DESC
""").fetchdf()

print(result)

          borough        room_type  num_listings  avg_price  avg_occupancy  \
0           Bronx  Entire home/apt           385     197.19          0.349   
1           Bronx     Private room           589     124.69          0.354   
2           Bronx      Shared room             2      53.42          0.527   
3        Brooklyn       Hotel room            43     385.72          0.238   
4        Brooklyn  Entire home/apt          5529     278.70          0.574   
5        Brooklyn     Private room          4889     147.01          0.508   
6        Brooklyn      Shared room           101     118.96          0.218   
7       Manhattan      Shared room            68     250.05          0.443   
8       Manhattan       Hotel room           384     818.00          0.258   
9       Manhattan  Entire home/apt          9035     399.33          0.461   
10      Manhattan     Private room          4222     297.95          0.542   
11         Queens       Hotel room            93     513.61     

##3.5  Pipeline Design & Automation

#Turn pipeline into configurable functions

In [56]:
import logging
from pathlib import Path
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger("airbnb_pipeline")

CONFIG = {
    "city": "new_york_city",
    "raw_dir": Path("../data/raw"),
    "interim_dir": Path("../data/interim"),
    "warehouse_path": Path("../data/airbnb_warehouse.duckdb"),
}

def ingest_city_data(config):
    """Stage 1: Load raw CSVs for a given city."""
    raw = config["raw_dir"]
    try:
        listings = pd.read_csv(raw / "listings2.csv")
        calendar = pd.read_csv(raw / "calendar.csv", parse_dates=["date"])
        reviews = pd.read_csv(raw / "reviews2.csv", parse_dates=["date"])
        neighbourhoods = pd.read_csv(raw / "neighbourhoods.csv")
        logger.info(f"[{config['city']}] Ingested: listings={len(listings):,}, "
                     f"calendar={len(calendar):,}, reviews={len(reviews):,}")
        return {"listings": listings, "calendar": calendar,
                "reviews": reviews, "neighbourhoods": neighbourhoods}
    except Exception as e:
        logger.error(f"[{config['city']}] Ingestion failed: {e}")
        raise

In [57]:
def clean_city_data(raw_data, config):
    """Stage 2: Apply cleaning/standardization rules."""
    listings = raw_data["listings"].copy()
    calendar = raw_data["calendar"].copy()

    dead_columns = [
        'neighborhood_overview', 'host_since', 'host_response_time',
        'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url',
        'host_neighbourhood', 'host_total_listings_count', 'host_verifications',
        'neighbourhood', 'calendar_updated', 'instant_bookable'
    ]
    listings = listings.drop(columns=[c for c in dead_columns if c in listings.columns])

    def clean_price(series):
        return series.astype(str).str.replace(r'[$,]', '', regex=True).replace('nan', pd.NA).astype(float)

    for col in ['price', 'price_quote_total_price', 'price_quote_price_per_night']:
        if col in listings.columns:
            listings[col] = clean_price(listings[col])

    SENTINEL = 2147483647
    calendar.loc[calendar['maximum_nights'] >= SENTINEL, 'maximum_nights'] = pd.NA
    for col in ['maximum_nights', 'maximum_maximum_nights', 'maximum_nights_avg_ntm']:
        if col in listings.columns:
            listings.loc[listings[col] >= SENTINEL, col] = pd.NA

    listings['has_reviews'] = listings['number_of_reviews'] > 0
    listings['first_review'] = pd.to_datetime(listings['first_review'], errors='coerce')
    listings['last_review'] = pd.to_datetime(listings['last_review'], errors='coerce')

    def simplify_property_type(pt):
        pt_lower = str(pt).lower()
        if 'entire' in pt_lower: return 'Entire place'
        elif 'private room' in pt_lower: return 'Private room'
        elif 'shared room' in pt_lower: return 'Shared room'
        elif 'hotel' in pt_lower or 'room in' in pt_lower: return 'Hotel/B&B'
        return 'Other'
    listings['property_type_group'] = listings['property_type'].apply(simplify_property_type)

    logger.info(f"[{config['city']}] Cleaning complete: {listings.shape[1]} columns retained")
    return {"listings": listings, "calendar": calendar,
            "reviews": raw_data["reviews"], "neighbourhoods": raw_data["neighbourhoods"]}

In [58]:
import json
from datetime import datetime

def save_stage_with_metadata(data_dict, stage_name, config):
    """Save cleaned data + a metadata record of what happened."""
    out_dir = config["interim_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)

    metadata = {
        "city": config["city"],
        "stage": stage_name,
        "run_timestamp": datetime.now().isoformat(),
        "tables": {}
    }

    for name, df in data_dict.items():
        path = out_dir / f"{name}_{stage_name}.parquet"
        df.to_parquet(path, index=False)
        metadata["tables"][name] = {"rows": len(df), "columns": df.shape[1], "path": str(path)}

    meta_path = out_dir / f"_metadata_{stage_name}_{config['city']}.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    logger.info(f"[{config['city']}] Stage '{stage_name}' saved with metadata -> {meta_path}")
    return metadata

In [59]:
raw_data = ingest_city_data(CONFIG)
cleaned_data = clean_city_data(raw_data, CONFIG)
metadata = save_stage_with_metadata(cleaned_data, "silver", CONFIG)

print(json.dumps(metadata, indent=2))

2026-07-11 11:38:04,742 | INFO | [new_york_city] Ingested: listings=30,259, calendar=11,152,576, reviews=990,170
2026-07-11 11:38:05,652 | INFO | [new_york_city] Cleaning complete: 80 columns retained
2026-07-11 11:38:11,597 | INFO | [new_york_city] Stage 'silver' saved with metadata -> ..\data\interim\_metadata_silver_new_york_city.json


{
  "city": "new_york_city",
  "stage": "silver",
  "run_timestamp": "2026-07-11T11:38:05.657222",
  "tables": {
    "listings": {
      "rows": 30259,
      "columns": 80,
      "path": "..\\data\\interim\\listings_silver.parquet"
    },
    "calendar": {
      "rows": 11152576,
      "columns": 5,
      "path": "..\\data\\interim\\calendar_silver.parquet"
    },
    "reviews": {
      "rows": 990170,
      "columns": 6,
      "path": "..\\data\\interim\\reviews_silver.parquet"
    },
    "neighbourhoods": {
      "rows": 230,
      "columns": 2,
      "path": "..\\data\\interim\\neighbourhoods_silver.parquet"
    }
  }
}


In [60]:
listings_final2.to_parquet("../data/interim/listings_final2_silver.parquet", index=False)